# Clinical Model Comparison — XGBoost vs. Logistic Regression vs. Random Forest

In [ ]:
import numpy as np
import pandas as pd
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, message=".*penalty.*was deprecated.*")
sys.path.insert(0, r"C:\FYP\src")
from utils.config import (
    TABULAR_CLEAN_PATH, CLINICAL_MODEL_COMPARISON_PATH, CLINICAL_OOF_PREDICTIONS_PATH,
    ensure_dirs, RANDOM_SEED,
)
from utils.metrics import early_stage_recall
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)
from xgboost import XGBClassifier

ensure_dirs()
pd.set_option("display.width", 120)
print("Imports OK")

Imports OK


## Load `tabular_clean.csv`

In [2]:
FEATURES = ["creatinine", "LYVE1", "REG1B", "TFF1", "plasma_CA19_9", "age", "sex"]
METADATA_COLS = ["sample_id", "patient_cohort", "sample_origin", "stage", "diagnosis"]
TARGET_COLS = ["dx", "target_binary"]

tabular_clean_df = pd.read_csv(TABULAR_CLEAN_PATH)
feature_matrix = tabular_clean_df[FEATURES].copy()
METADATA = tabular_clean_df[METADATA_COLS].copy()
TARGETS = tabular_clean_df[TARGET_COLS].copy()

print(f"feature_matrix: {feature_matrix.shape}   METADATA: {METADATA.shape}   TARGETS: {TARGETS.shape}")
print(f"plasma_CA19_9 missing: {feature_matrix['plasma_CA19_9'].isna().sum()}")

feature_matrix: (590, 7)   METADATA: (590, 5)   TARGETS: (590, 2)
plasma_CA19_9 missing: 240


## The Settled Imputer: `MICE_CA19_9Imputer`

In [3]:
class MICE_CA19_9Imputer:
    PREDICTORS = ["creatinine", "LYVE1", "REG1B", "TFF1", "age"]
    TARGET = "plasma_CA19_9"

    def __init__(self, random_state=RANDOM_SEED):
        self.imputer = IterativeImputer(estimator=BayesianRidge(), random_state=random_state)

    def fit(self, train_df):
        self.imputer.fit(train_df[self.PREDICTORS + [self.TARGET]])
        return self

    def transform(self, target_df):
        out = target_df.copy()
        out[self.TARGET] = self.imputer.transform(out[self.PREDICTORS + [self.TARGET]])[:, -1]
        return out


print("MICE_CA19_9Imputer defined.")

MICE_CA19_9Imputer defined.


## Three Candidate Models

In [4]:
MODELS = {
    "XGBoost": lambda: XGBClassifier(
        n_estimators=100, max_depth=3, eval_metric="logloss", random_state=0
    ),
    "LogisticRegression": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=0)),
    ]),
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=200, max_depth=5, random_state=0
    ),
}
print(f"Models defined: {list(MODELS.keys())}")

Models defined: ['XGBoost', 'LogisticRegression', 'RandomForest']


## Early-Stage Definition for This Notebook

In [5]:
EARLY_STAGES_NOTEBOOK = {"I", "IA", "IB", "II", "IIA"}
n_early_total = METADATA["stage"].isin(EARLY_STAGES_NOTEBOOK).sum()
print(f"Early-stage patients under this definition: {n_early_total} / {len(METADATA)}")
print(METADATA[METADATA['stage'].isin(EARLY_STAGES_NOTEBOOK)]['stage'].value_counts())

Early-stage patients under this definition: 34 / 590
stage
IB     12
IIA    11
II      7
IA      3
I       1
Name: count, dtype: int64


## `run_fold` — Full Metric Suite, MICE Fixed, Model Swappable

In [ ]:
def run_fold(
    train_idx, test_idx, model_name, model_factory,
    feature_matrix, targets, metadata,
    oof_rows=None, repeat_idx=None, cv_fold_idx=None,
):
    train_fold, test_fold = feature_matrix.loc[train_idx], feature_matrix.loc[test_idx]
    y_train, y_test = targets.loc[train_idx, "target_binary"], targets.loc[test_idx, "target_binary"]

    imputer = MICE_CA19_9Imputer().fit(train_fold)
    train_imp, test_imp = imputer.transform(train_fold), imputer.transform(test_fold)

    model = model_factory()
    model.fit(train_imp[FEATURES], y_train)
    proba = model.predict_proba(test_imp[FEATURES])[:, 1]
    preds = model.predict(test_imp[FEATURES])
    preds_series = pd.Series(preds, index=test_idx)

    multi_class = y_test.nunique() > 1
    auc = roc_auc_score(y_test, proba) if multi_class else np.nan

    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
    n_pos_true = int((y_test == 1).sum())
    n_pos_pred = int((preds == 1).sum())
    
    recall = recall_score(y_test, preds, zero_division=0) if n_pos_true > 0 else np.nan  # sensitivity
    precision = precision_score(y_test, preds, zero_division=0) if n_pos_pred > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = f1_score(y_test, preds, zero_division=0) if (n_pos_true > 0 and n_pos_pred > 0) else np.nan
    acc = accuracy_score(y_test, preds)

    stage_test = metadata.loc[test_idx, "stage"]
    early_recall = early_stage_recall(y_test, preds, stage_test, early_stages=EARLY_STAGES_NOTEBOOK)
    n_early = int(stage_test.isin(EARLY_STAGES_NOTEBOOK).sum())

    if oof_rows is not None:
        for pos, idx in enumerate(test_idx):
            oof_rows.append({
                "repeat_idx": repeat_idx, "cv_fold_idx": cv_fold_idx, "model": model_name,
                "sample_id": metadata.loc[idx, "sample_id"],
                "y_true": int(y_test.loc[idx]), "y_proba": float(proba[pos]),
                "y_pred": int(preds_series.loc[idx]),
            })

    return {
        "auc": auc, "precision": precision, "recall": recall, "specificity": specificity,
        "f1": f1, "acc": acc, "early_recall": early_recall, "n_early": n_early,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


print("run_fold() defined.")

run_fold() defined.


### Repeated Stratified 5-Fold × 20-Repeat CV — All Three Models, Same Folds

In [7]:
comparison_rows = []
oof_rows = []

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_SEED)
repeated_cv_splits = [
    (feature_matrix.index[train_pos], feature_matrix.index[test_pos])
    for train_pos, test_pos in rskf.split(feature_matrix, TARGETS["target_binary"])
]
print(f"Precomputed {len(repeated_cv_splits)} repeated-CV folds (shared across all 3 models).\n")


def summarize(fold_df, scheme, model_name):
    return {
        "scheme": scheme, "model": model_name, "n_folds": len(fold_df),
        "precision_mean": fold_df["precision"].mean(), "precision_std": fold_df["precision"].std(),
        "recall_mean": fold_df["recall"].mean(), "recall_std": fold_df["recall"].std(),
        "specificity_mean": fold_df["specificity"].mean(skipna=True), "specificity_std": fold_df["specificity"].std(skipna=True),
        "f1_mean": fold_df["f1"].mean(), "f1_std": fold_df["f1"].std(),
        "auc_mean": fold_df["auc"].mean(skipna=True), "auc_std": fold_df["auc"].std(skipna=True),
        "early_recall_mean": fold_df["early_recall"].mean(skipna=True), "early_recall_std": fold_df["early_recall"].std(skipna=True),
        "early_recall_n_total": int(fold_df["n_early"].sum()),
        "acc_mean": fold_df["acc"].mean(), "acc_std": fold_df["acc"].std(),
        "tn": int(fold_df["tn"].sum()), "fp": int(fold_df["fp"].sum()),
        "fn": int(fold_df["fn"].sum()), "tp": int(fold_df["tp"].sum()),
    }


for model_name, model_factory in MODELS.items():
    fold_results = []
    for fold_i, (train_idx, test_idx) in enumerate(repeated_cv_splits):
        repeat_idx, cv_fold_idx = divmod(fold_i, 5)
        result = run_fold(
            train_idx, test_idx, model_name, model_factory, feature_matrix, TARGETS, METADATA,
            oof_rows=oof_rows, repeat_idx=repeat_idx, cv_fold_idx=cv_fold_idx,
        )
        fold_results.append(result)
    fold_df = pd.DataFrame(fold_results)
    comparison_rows.append(summarize(fold_df, "Repeated 5x20 CV", model_name))
    print(f"[{model_name}] AUC {fold_df['auc'].mean():.4f}   Recall(sens) {fold_df['recall'].mean():.4f}   "
          f"Specificity {fold_df['specificity'].mean():.4f}   F1 {fold_df['f1'].mean():.4f}   "
          f"Early recall {fold_df['early_recall'].mean(skipna=True):.4f} (n={int(fold_df['n_early'].sum())})")

Precomputed 100 repeated-CV folds (shared across all 3 models).



[XGBoost] AUC 0.9077   Recall(sens) 0.7433   Specificity 0.8884   F1 0.7573   Early recall 0.6379 (n=680)


[LogisticRegression] AUC 0.8786   Recall(sens) 0.6157   Specificity 0.9130   F1 0.6884   Early recall 0.5168 (n=680)


[RandomForest] AUC 0.9068   Recall(sens) 0.6920   Specificity 0.9060   F1 0.7370   Early recall 0.6109 (n=680)


### Cohort-Out Split — All Three Models, Same Split

In [8]:
cohort_out_train_idx = METADATA.index[METADATA["patient_cohort"] == "Cohort1"]
cohort_out_test_idx = METADATA.index[METADATA["patient_cohort"] == "Cohort2"]

for model_name, model_factory in MODELS.items():
    result = run_fold(
        cohort_out_train_idx, cohort_out_test_idx, model_name, model_factory,
        feature_matrix, TARGETS, METADATA,
    )
    fold_df = pd.DataFrame([result])
    comparison_rows.append(summarize(fold_df, "Cohort-out", model_name))
    print(f"[{model_name}] AUC {result['auc']:.4f}   Recall(sens) {result['recall']:.4f}   "
          f"Specificity {result['specificity']:.4f}   F1 {result['f1']:.4f}   "
          f"Early recall {result['early_recall']:.4f} (n={result['n_early']})")

[XGBoost] AUC 0.8375   Recall(sens) 0.6757   Specificity 0.9005   F1 0.5952   Early recall 0.5000 (n=10)
[LogisticRegression] AUC 0.8793   Recall(sens) 0.7027   Specificity 0.8959   F1 0.6047   Early recall 0.5000 (n=10)


[RandomForest] AUC 0.8901   Recall(sens) 0.7568   Specificity 0.9005   F1 0.6437   Early recall 0.6000 (n=10)


### Leave-One-Site-Out — All Three Models, Same Splits

In [9]:
site_splits = {
    site: (METADATA.index[METADATA["sample_origin"] != site], METADATA.index[METADATA["sample_origin"] == site])
    for site in sorted(METADATA["sample_origin"].unique())
}

site_detail_rows = []
for model_name, model_factory in MODELS.items():
    per_site = []
    for site, (train_idx, test_idx) in site_splits.items():
        result = run_fold(train_idx, test_idx, model_name, model_factory, feature_matrix, TARGETS, METADATA)
        per_site.append(result)
        site_detail_rows.append({"model": model_name, "site": site, "n_test": len(test_idx), **result})
    fold_df = pd.DataFrame(per_site)
    comparison_rows.append(summarize(fold_df, "Leave-one-site-out", model_name))
    print(f"[{model_name}] Mean AUC {fold_df['auc'].mean(skipna=True):.4f}   "
          f"Mean recall(sens) {fold_df['recall'].mean():.4f}   "
          f"Mean early recall {fold_df['early_recall'].mean(skipna=True):.4f}")

site_detail_df = pd.DataFrame(site_detail_rows)
print("\nPer-site detail:")
display(site_detail_df[["model", "site", "n_test", "auc", "recall", "specificity", "f1", "early_recall", "n_early"]])

[XGBoost] Mean AUC 0.8195   Mean recall(sens) 0.7783   Mean early recall 0.7516


[LogisticRegression] Mean AUC 0.8221   Mean recall(sens) 0.7483   Mean early recall 0.6906


[RandomForest] Mean AUC 0.8324   Mean recall(sens) 0.7290   Mean early recall 0.6558

Per-site detail:


,model,site,n_test,auc,recall,specificity,f1,early_recall,n_early
0,XGBoost,BPTB,409,0.842006,0.722892,0.782209,0.560748,0.666667,9
1,XGBoost,ESP,29,0.782609,0.913043,0.333333,0.875000,1.000000,8
2,XGBoost,LIV,132,0.833747,0.698925,0.794872,0.783133,0.588235,17
3,XGBoost,UCL,20,NaN,NaN,0.850000,NaN,NaN,0
4,LogisticRegression,BPTB,409,0.819018,0.783133,0.650307,0.496183,0.777778,9
5,LogisticRegression,ESP,29,0.826087,0.956522,0.500000,0.916667,1.000000,8
6,LogisticRegression,LIV,132,0.821340,0.505376,0.846154,0.643836,0.294118,17
7,LogisticRegression,UCL,20,NaN,NaN,0.800000,NaN,NaN,0
8,RandomForest,BPTB,409,0.845184,0.746988,0.757669,0.553571,0.555556,9
9,RandomForest,ESP,29,0.840580,0.913043,0.666667,0.913043,1.000000,8


## Comparison Table — All Metrics, All (Model × Scheme) Cells

In [10]:
comparison_df = pd.DataFrame(comparison_rows)[[
    "scheme", "model", "n_folds",
    "precision_mean", "precision_std", "recall_mean", "recall_std",
    "specificity_mean", "specificity_std", "f1_mean", "f1_std",
    "auc_mean", "auc_std", "early_recall_mean", "early_recall_std", "early_recall_n_total",
    "acc_mean", "acc_std", "tn", "fp", "fn", "tp",
]].round(4)

print("Full scheme x model comparison:")
display(comparison_df)

Full scheme x model comparison:


,scheme,model,n_folds,precision_mean,precision_std,recall_mean,recall_std,specificity_mean,specificity_std,f1_mean,...,auc_std,early_recall_mean,early_recall_std,early_recall_n_total,acc_mean,acc_std,tn,fp,fn,tp
0,Repeated 5x20 CV,XGBoost,100,0.7760,0.0578,0.7433,0.0554,0.8884,0.0364,0.7573,...,0.0224,0.6379,0.1965,680,0.8394,0.0277,6947,873,1022,2958
1,Repeated 5x20 CV,LogisticRegression,100,0.7877,0.0641,0.6157,0.0609,0.9130,0.0330,0.6884,...,0.0304,0.5168,0.2089,680,0.8127,0.0266,7140,680,1530,2450
2,Repeated 5x20 CV,RandomForest,100,0.7943,0.0662,0.6920,0.0632,0.9060,0.0361,0.7370,...,0.0239,0.6109,0.1930,680,0.8338,0.0294,7085,735,1226,2754
3,Cohort-out,XGBoost,1,0.5319,NaN,0.6757,NaN,0.9005,NaN,0.5952,...,NaN,0.5000,NaN,10,0.8682,NaN,199,22,12,25
4,Cohort-out,LogisticRegression,1,0.5306,NaN,0.7027,NaN,0.8959,NaN,0.6047,...,NaN,0.5000,NaN,10,0.8682,NaN,198,23,11,26
5,Cohort-out,RandomForest,1,0.5600,NaN,0.7568,NaN,0.9005,NaN,0.6437,...,NaN,0.6000,NaN,10,0.8798,NaN,199,22,9,28
6,Leave-one-site-out,XGBoost,4,0.5471,0.4127,0.7783,0.1173,0.6901,0.2397,0.7396,...,0.0322,0.7516,0.2186,34,0.7851,0.0511,305,86,53,146
7,Leave-one-site-out,LogisticRegression,4,0.5325,0.4315,0.7483,0.2276,0.6991,0.1569,0.6856,...,0.0036,0.6906,0.3609,34,0.7363,0.1159,264,127,65,134
8,Leave-one-site-out,RandomForest,4,0.5569,0.4289,0.7290,0.1937,0.7487,0.0632,0.7081,...,0.0183,0.6558,0.3067,34,0.7453,0.1018,298,93,67,132


## Sensitivity (Recall) — Dedicated Section

In [11]:
sensitivity_table = comparison_df.pivot(index="scheme", columns="model", values="recall_mean")
print("Sensitivity (Recall) -- mean across folds per scheme, by model:")
display(sensitivity_table)

Sensitivity (Recall) -- mean across folds per scheme, by model:


model,LogisticRegression,RandomForest,XGBoost
scheme,,,
Cohort-out,0.7027,0.7568,0.6757
Leave-one-site-out,0.7483,0.7290,0.7783
Repeated 5x20 CV,0.6157,0.6920,0.7433


## Early-Stage Recall — Dedicated Section (Directional Only)

In [12]:
early_recall_table = comparison_df.pivot(index="scheme", columns="model", values="early_recall_mean")
early_recall_n = comparison_df.pivot(index="scheme", columns="model", values="early_recall_n_total")
print("Early-stage recall -- mean across folds per scheme, by model:")
display(early_recall_table)
print("\nTotal early-stage patient-instances scored (sum across folds) -- sample size behind the above:")
display(early_recall_n)

Early-stage recall -- mean across folds per scheme, by model:


model,LogisticRegression,RandomForest,XGBoost
scheme,,,
Cohort-out,0.5000,0.6000,0.5000
Leave-one-site-out,0.6906,0.6558,0.7516
Repeated 5x20 CV,0.5168,0.6109,0.6379



Total early-stage patient-instances scored (sum across folds) -- sample size behind the above:


model,LogisticRegression,RandomForest,XGBoost
scheme,,,
Cohort-out,10,10,10
Leave-one-site-out,34,34,34
Repeated 5x20 CV,680,680,680


## Write `results/clinical/model_comparison.csv` and `oof_predictions.csv`

In [13]:
comparison_df.to_csv(CLINICAL_MODEL_COMPARISON_PATH, index=False)
print(f"Wrote {CLINICAL_MODEL_COMPARISON_PATH} ({len(comparison_df)} rows)")

oof_df = pd.DataFrame(oof_rows)
oof_df.to_csv(CLINICAL_OOF_PREDICTIONS_PATH, index=False)
print(f"Wrote {CLINICAL_OOF_PREDICTIONS_PATH} ({len(oof_df):,} rows)")

Wrote C:\FYP\results\clinical\model_comparison.csv (9 rows)


Wrote C:\FYP\results\clinical\oof_predictions.csv (35,400 rows)


## ROC, Precision-Recall, and Confusion Matrix -- All Three Models

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
from utils.config import EVAL_CLINICAL_DIR

MODEL_COLORS = {"XGBoost": "C0", "LogisticRegression": "C1", "RandomForest": "C2"}
n_repeats = oof_df["repeat_idx"].nunique()

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 6, height_ratios=[1.15, 1])
ax_roc = fig.add_subplot(gs[0, 0:3])
ax_pr = fig.add_subplot(gs[0, 3:6])
cm_axes = [fig.add_subplot(gs[1, i * 2:i * 2 + 2]) for i in range(3)]

for model_name in MODELS:
    sub = oof_df[oof_df["model"] == model_name]
    y_true, y_proba = sub["y_true"].to_numpy(), sub["y_proba"].to_numpy()
    color = MODEL_COLORS[model_name]

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    ax_roc.plot(fpr, tpr, color=color, label=f"{model_name} (AUC={roc_auc_score(y_true, y_proba):.4f})")

    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    ax_pr.plot(rec, prec, color=color, label=model_name)

ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax_roc.set_xlabel("False Positive Rate"); ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title("ROC (pooled OOF, Repeated 5x20 CV)"); ax_roc.legend(loc="lower right")

ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")
ax_pr.set_title("Precision-Recall (pooled OOF)"); ax_pr.legend(loc="lower left")

for ax, model_name in zip(cm_axes, MODELS):
    sub = oof_df[oof_df["model"] == model_name]
    cm_summed = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
    cm_avg = cm_summed / n_repeats
    cm_pct = cm_avg / cm_avg.sum(axis=1, keepdims=True) * 100

    ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm_pct[i, j]:.1f}%\n(n={cm_avg[i, j]:.1f})", ha="center", va="center",
                    color="white" if cm_pct[i, j] > 55 else "black", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Benign", "PDAC"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Benign", "PDAC"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{model_name}\n(threshold=0.5, mean of {n_repeats} repeats)")

fig.suptitle(
    "Clinical Model Comparison -- ROC, Precision-Recall, Confusion Matrix\n"
    "(Repeated 5x20 CV, pooled/averaged out-of-fold predictions)", fontsize=13
)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig_path = EVAL_CLINICAL_DIR / "model_comparison_curves.png"
fig.savefig(fig_path, dpi=100)
plt.close(fig)
print(f"Saved {fig_path}")

Saved C:\FYP\outputs\eval\clinical\model_comparison_curves.png


## Threshold Sweep and Calibration -- Is 0.5 the Right Cutoff?

In [15]:
from sklearn.metrics import brier_score_loss

thresholds = np.linspace(0.01, 0.99, 99)

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.05])
sweep_axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
ax_reliability = fig.add_subplot(gs[1, :])

for ax, model_name in zip(sweep_axes, MODELS):
    sub = oof_df[oof_df["model"] == model_name]
    y_true, y_proba = sub["y_true"].to_numpy(), sub["y_proba"].to_numpy()

    precisions, recalls, specificities, f1s = [], [], [], []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())
        precisions.append(tp / (tp + fp) if (tp + fp) > 0 else np.nan)
        recalls.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
        f1_denom = (2 * tp + fp + fn)
        f1s.append(2 * tp / f1_denom if f1_denom > 0 else np.nan)

    ax.plot(thresholds, precisions, label="Precision")
    ax.plot(thresholds, recalls, label="Recall (sensitivity)")
    ax.plot(thresholds, specificities, label="Specificity")
    ax.plot(thresholds, f1s, label="F1")
    ax.axvline(0.5, color="k", linestyle="--", alpha=0.3, label="threshold=0.5 (used elsewhere)")
    ax.set_xlabel("Decision threshold"); ax.set_ylabel("Score")
    ax.set_title(model_name)
    ax.set_ylim(0, 1.02)
    if model_name == list(MODELS.keys())[0]:
        ax.legend(fontsize=8, loc="lower left")

for model_name in MODELS:
    sub = oof_df[oof_df["model"] == model_name]
    y_true, y_proba = sub["y_true"].to_numpy(), sub["y_proba"].to_numpy()
    color = MODEL_COLORS[model_name]

    bin_edges = np.linspace(0, 1, 11)
    bin_ids = np.clip(np.digitize(y_proba, bin_edges) - 1, 0, 9)
    bin_obs = [y_true[bin_ids == i].mean() if (bin_ids == i).any() else np.nan for i in range(10)]
    bin_pred = [y_proba[bin_ids == i].mean() if (bin_ids == i).any() else np.nan for i in range(10)]
    brier = brier_score_loss(y_true, y_proba)
    ax_reliability.plot(bin_pred, bin_obs, marker="o", color=color, label=f"{model_name} (Brier={brier:.4f})")

ax_reliability.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax_reliability.set_xlabel("Mean predicted probability"); ax_reliability.set_ylabel("Observed frequency")
ax_reliability.set_title("Reliability diagram (UNCALIBRATED, pooled OOF, Repeated 5x20 CV) -- all 3 candidates")
ax_reliability.legend(loc="upper left")

fig.suptitle(
    "Clinical Model Comparison -- Threshold Sweep and Calibration (pooled OOF, Repeated 5x20 CV)\n"
    "Note: XGBoost's calibrated (Platt-scaled) reliability curve for the final promoted model is "
    "shown separately in outputs/eval/fusion/calibration_curves.png", fontsize=12,
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig_path = EVAL_CLINICAL_DIR / "threshold_and_calibration.png"
fig.savefig(fig_path, dpi=100)
plt.close(fig)
print(f"Saved {fig_path}")

Saved C:\FYP\outputs\eval\clinical\threshold_and_calibration.png


## Consistency Check Against `clinical_imputer_benchmark.ipynb`

In [16]:
xgb_check = comparison_df[comparison_df.model == "XGBoost"].set_index("scheme")["auc_mean"]

# Cohort-out and leave-one-site-out don't depend on n_repeats at all (single, deterministic
# splits) -- these stay hard-checked against the original verified numbers.
known_good = {"Cohort-out": 0.8375, "Leave-one-site-out": 0.8195}
for scheme, expected in known_good.items():
    actual = xgb_check.loc[scheme]
    print(f"  {scheme:22s} expected={expected:.4f}  actual={actual:.4f}  "
          f"[{'MATCH' if abs(actual - expected) < 0.001 else 'DIFFERS'}]")

# Repeated CV now runs 20 repeats (was 10) -- a different, tighter estimate of the same
# quantity by design, so it is NOT expected to exactly match the 10-repeat figure (0.9081).
# Reported here for information, not as a pass/fail check.
print(f"  {'Repeated 5x20 CV':22s} (informational, not a fixed target) actual="
      f"{xgb_check.loc['Repeated 5x20 CV']:.4f}")

  Cohort-out             expected=0.8375  actual=0.8375  [MATCH]
  Leave-one-site-out     expected=0.8195  actual=0.8195  [MATCH]
  Repeated 5x20 CV       (informational, not a fixed target) actual=0.9077


## Statistical Significance -- XGBoost vs. Random Forest

In [17]:
from scipy.stats import wilcoxon


def per_fold_metric(df, model, metric_fn):
    vals = []
    for (r, f), g in df[df.model == model].groupby(['repeat_idx', 'cv_fold_idx']):
        vals.append(metric_fn(g.y_true, g.y_pred))
    return np.array(vals)


xgb_f1 = per_fold_metric(oof_df, 'XGBoost', f1_score)
rf_f1  = per_fold_metric(oof_df, 'RandomForest', f1_score)
xgb_recall = per_fold_metric(oof_df, 'XGBoost', recall_score)
rf_recall  = per_fold_metric(oof_df, 'RandomForest', recall_score)

stat_f1, p_f1 = wilcoxon(xgb_f1, rf_f1)
stat_recall, p_recall = wilcoxon(xgb_recall, rf_recall)

print(f"Per-fold F1:     XGBoost mean={xgb_f1.mean():.4f}  RandomForest mean={rf_f1.mean():.4f}  "
      f"(n={len(xgb_f1)} folds)")
print(f"Wilcoxon signed-rank (F1):     statistic={stat_f1:.1f}, p={p_f1:.6f}")
print()
print(f"Per-fold Recall: XGBoost mean={xgb_recall.mean():.4f}  RandomForest mean={rf_recall.mean():.4f}  "
      f"(n={len(xgb_recall)} folds)")
print(f"Wilcoxon signed-rank (Recall): statistic={stat_recall:.1f}, p={p_recall:.6f}")


Per-fold F1:     XGBoost mean=0.7573  RandomForest mean=0.7370  (n=100 folds)
Wilcoxon signed-rank (F1):     statistic=1174.0, p=0.000015

Per-fold Recall: XGBoost mean=0.7433  RandomForest mean=0.6920  (n=100 folds)
Wilcoxon signed-rank (Recall): statistic=396.5, p=0.000000


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

rep0 = oof_df[oof_df.repeat_idx == 0]
xgb = rep0[rep0.model == 'XGBoost'].set_index('sample_id').sort_index()
rf  = rep0[rep0.model == 'RandomForest'].set_index('sample_id').sort_index()

assert (xgb.index == rf.index).all(), "XGBoost/RandomForest OOF samples don't align for repeat 0"

xgb_correct = (xgb.y_pred.values == xgb.y_true.values)
rf_correct  = (rf.y_pred.values == rf.y_true.values)

table = [
    [int((xgb_correct & rf_correct).sum()),  int((xgb_correct & ~rf_correct).sum())],
    [int((~xgb_correct & rf_correct).sum()), int((~xgb_correct & ~rf_correct).sum())],
]
result = mcnemar(table, exact=False)

print(f"McNemar contingency table (repeat 0, n={len(xgb)} paired samples):")
print(f"  both correct            = {table[0][0]}")
print(f"  only XGBoost correct     = {table[0][1]}")
print(f"  only RandomForest correct= {table[1][0]}")
print(f"  both wrong               = {table[1][1]}")
print(f"\nMcNemar (chi-square approximation): statistic={result.statistic:.4f}, p={result.pvalue:.6f}")


McNemar contingency table (repeat 0, n=590 paired samples):
  both correct            = 467
  only XGBoost correct     = 36
  only RandomForest correct= 28
  both wrong               = 59

McNemar (chi-square approximation): statistic=0.7656, p=0.381574
